In [ ]:
# ==========================================
# BƯỚC 0: KẾT NỐI DRIVE & CÀI ĐẶT
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics huggingface_hub -q

from ultralytics import YOLO
from huggingface_hub import login, hf_hub_download
import os

login("Nhập token")
# ==========================================
# BƯỚC 1: KÉO DỮ LIỆU TỪ HUGGING FACE (CHỈ LẤY TẬP TEST)
# ==========================================
REPO_ID = "ToQuangHan/tsbow"

print("Đang tải tập Test...")
# Chú ý: Đảm bảo trên Hugging Face của bạn có file này nhé
test_zip = hf_hub_download(repo_id=REPO_ID, filename="annotations_test.zip", repo_type="dataset")

# ==========================================
# BƯỚC 2: GIẢI NÉN VÀO CẤU TRÚC YOLO
# ==========================================
print("Đang giải nén dữ liệu Test...")
!mkdir -p /content/datasets/TSBOW/test
!unzip -o -q {test_zip} -d /content/datasets/TSBOW/test

# ==========================================
# BƯỚC 3: TẠO FILE TSBOW.YAML
# ==========================================
yaml_content = """
path: /content/datasets/TSBOW
train: test
val: test
test: test  # Lần này chỉ cần khai báo test là đủ để chấm điểm

nc: 8
names:
  0: 'unidentified'
  1: 'others'
  2: 'pedestrian'
  3: 'micromobility'
  4: 'car'
  5: 'bus'
  6: 'small truck'
  7: 'truck'
"""
with open('/content/datasets/TSBOW/TSBOW.yaml', 'w') as f:
    f.write(yaml_content)

print("Đã chuẩn bị xong Data Test!")

# ==========================================
# BƯỚC 4: TIẾN HÀNH THI ĐẠI HỌC (TEST)
# ==========================================

# 1. Đường dẫn tới file weight TỐT NHẤT (best.pt)
best_weight_path = "/content/drive/MyDrive/TSBOW_Results/tsbow_run/weights/best.pt"

if os.path.exists(best_weight_path):
    print("✅ Đã tìm thấy file best.pt! Bắt đầu nạp mô hình...")
    model = YOLO(best_weight_path)

    print("🚀 Đang chấm điểm trên tập Test...")
    # 2. Chạy lệnh val nhưng ép nó dùng split='test'
    metrics = model.val(
    data='/content/datasets/TSBOW/TSBOW.yaml',
    split='test',
    project='/content/drive/MyDrive/TSBOW_Results', # Đường dẫn thư mục gốc trên Drive
    name='test_results'                             # Tên thư mục con sẽ chứa kết quả
)

    print(f"🎉 Điểm mAP50 chính thức trên tập Test đạt: {metrics.box.map50:.3f}")
else:
    print("❌ Không tìm thấy file best.pt trên Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đang tải tập Test...
Đang giải nén dữ liệu Test...
Đã chuẩn bị xong Data Test!
✅ Đã tìm thấy file best.pt! Bắt đầu nạp mô hình...
🚀 Đang chấm điểm trên tập Test...
Ultralytics 8.4.65 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,036,200 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2361.6±327.8 MB/s, size: 260.3 KB)
val: Scanning /content/datasets/TSBOW/test/labels... 6614 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 6614/6614 1.8Kit/s 3.7s
val: /content/datasets/TSBOW/test/images/suwon#11_01_04_006540.jpg: 1 duplicate labels removed
val: /content/datasets/TSBOW/test/images/suwon#11b_01_01_008325.jpg: 1 duplicate labels removed
val: /content/datasets/TSBOW/test/images/suwon#11d_06_01_007275.jpg: 1 duplicate labels removed
val: /content/datasets/TSBOW/test